In [17]:
import pandas as pd
import numpy as np
import os

In [18]:
itjobs = pd.read_csv('itjobspoland.csv')
itjobs.isnull().sum()

Title                           0
City                            0
Country_code                   33
Marker_icon                     0
Workplace_type                  0
Experience_level                0
Published_at                    0
Remote_interview                0
Remote                          0
Open_to_hire_Ukrainians      5202
Company_size_from              55
Company_size_to                39
if_permanent                    0
salary_from_permanent           0
salary_to_permanent             0
salary_currency_permanent       0
if_b2b                          0
salary_from_b2b                 0
salary_to_b2b                   0
salary_currency_b2b             0
if_mandate                      0
salary_from_mandate             0
salary_to_mandate               0
salary_currency_mandate         0
if_other                        0
salary_from_other               0
salary_to_other                 0
salary_currency_other           0
currency_exchange_rate          0
skills_name_0 

In [19]:

df = itjobs.dropna(subset=['skills_name_0','skills_name_2'])

df['Country_code'] = df['Country_code'].fillna('PL')
print(f"Amount of offers: {len(df)}")
df = df.drop(columns=['Open_to_hire_Ukrainians'], errors='ignore')
print(df.dtypes)

salary_cols = [
    'salary_from_permanent', 'salary_to_permanent',
    'salary_from_b2b', 'salary_to_b2b',
    'salary_from_mandate', 'salary_to_mandate',
    'salary_from_other', 'salary_to_other',
    'currency_exchange_rate'
]

df[salary_cols] = df[salary_cols].fillna(0.0)

for col in ['salary_from_permanent', 'salary_to_permanent', 'salary_from_b2b', 'salary_to_b2b']:
    df[col] = df[col].astype(int)

Amount of offers: 37721
Title                         object
City                          object
Country_code                  object
Marker_icon                   object
Workplace_type                object
Experience_level              object
Published_at                  object
Remote_interview                bool
Remote                          bool
Company_size_from             object
Company_size_to               object
if_permanent                    bool
salary_from_permanent        float64
salary_to_permanent          float64
salary_currency_permanent     object
if_b2b                          bool
salary_from_b2b              float64
salary_to_b2b                float64
salary_currency_b2b           object
if_mandate                      bool
salary_from_mandate          float64
salary_to_mandate            float64
salary_currency_mandate       object
if_other                        bool
salary_from_other              int64
salary_to_other                int64
salary_currenc

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26812\284750464.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Country_code'] = df['Country_code'].fillna('PL')


In [20]:

#Experience distribution
print(df['Experience_level'].value_counts(dropna=False))

#Workplace distribution
print(df['Workplace_type'].value_counts(dropna=False))

#top 10 cities
print(df['City'].value_counts().head(10))


#outliers
b2b_salaries = df[df['salary_from_b2b'] > 0]['salary_from_b2b']
Q1 = b2b_salaries.quantile(0.25)
Q3 = b2b_salaries.quantile(0.75)
IQR = Q3 - Q1
lo = Q1 - 1.5 * IQR
up = Q3 + 1.5 * IQR

print(f"Lower bound IQR: {lo}")
print(f"Upper bound IQR: {up}")

outliers_b2b = df[
    ((df['salary_from_b2b'] > upper_bound) | 
     ((df['salary_from_b2b'] < lower_bound) & (df['salary_from_b2b'] > 0)))
]
print(f"\nAmount of outliers: {len(outliers_b2b)}")

if len(outliers_b2b) > 0:
    print(outliers_b2b[['Title', 'City', 'Experience_level', 'salary_from_b2b', 'salary_currency_b2b']]
          .sort_values(by='salary_from_b2b', ascending=False)
          .head(10))


Experience_level
mid       20070
senior    12451
junior     5200
Name: count, dtype: int64
Workplace_type
remote           24431
partly_remote    11412
office            1878
Name: count, dtype: int64
City
Warszawa     13873
Kraków        5827
Wrocław       4798
Gdańsk        2605
Poznań        2181
Katowice      1357
Łódź          1042
Białystok      558
Gdynia         525
Gliwice        479
Name: count, dtype: int64
Lower bound IQR: -2500.0
Upper bound IQR: 33500.0

Amount of outliers: 298
                                   Title       City Experience_level  \
530    PostgreSQL Database Administrator    Wrocław           senior   
11200                C++ Group Interview    Wrocław              mid   
19658      Tech Lead Manager (Front-End)   Warszawa           senior   
19671    Staff Backend Software Engineer    Rzeszów           senior   
19670  Staff Fullstack Software Engineer  Bydgoszcz           senior   
19661               Data Science Manager     Kraków           senior   

In [21]:
#quick check if salaries are switched
print(len(df[df['salary_from_b2b'] > df['salary_to_b2b']]))
print(len(df[df['salary_from_permanent'] > df['salary_to_permanent']]))

0
0


In [22]:
#saving the changed dataset
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
output_filepath = os.path.join(desktop_path, "itjobs.csv")
df.to_csv(output_filepath, index=False, encoding='utf-8-sig')